# 07 Results Freeze

Final reviewer evidence matrix and artifact freeze.

## Setup

Clone/pull latest code into `/content/ECG-RAMBA`, restore verified Drive mirror artifacts, and define a logging command helper.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')

os.chdir('/content')
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
if not REPO_DIR.exists():
    print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
    subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

os.chdir(REPO_DIR)

def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    print(f'Command log: {log_path}')
    if check and rc:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(rc, cmd)
    return subprocess.CompletedProcess(cmd, rc)

run('git fetch origin', check=False)
run(f'git checkout {BRANCH}', check=True)
run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)

if MIRROR_REVISION_ROOT.exists():
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_restore.log',
    )
else:
    print('Mirror not found; continuing with repo-local reports/revision:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
run('git rev-parse --short HEAD', check=False)
run('git status --short --branch', check=False)


## Required Input Check

Validate the revision-plan CSV files and the final evidence inputs before building the final matrix. This fails early if paired ResNet or other required artifacts are missing from the restored mirror.


In [ ]:
import pandas as pd

revision_root = Path('reports/revision')
plan_csvs = [
    Path('docs/revision_plan/experiment_registry.csv'),
    Path('docs/revision_plan/claim_evidence_map.csv'),
    Path('docs/revision_plan/task_board.csv'),
]
for csv_path in plan_csvs:
    df = pd.read_csv(csv_path)
    print(f'{csv_path}: rows={len(df)} cols={len(df.columns)}')

required_final_inputs = {
    'calibration': revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json',
    'pooling': revision_root / 'metrics' / 'pooling_sensitivity.csv',
    'baseline': revision_root / 'metrics' / 'baseline_summary.csv',
    'component': revision_root / 'metrics' / 'component_check_summary.json',
    'hrv_domain': revision_root / 'metrics' / 'hrv_domain_summary.csv',
    'robustness': revision_root / 'metrics' / 'robustness_summary.csv',
    'paired_minirocket': revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    'paired_resnet': revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
    'a0_status': revision_root / 'a0_resolution_status.json',
    'claim_map': Path('docs/revision_plan/claim_evidence_map.csv'),
    'task_board': Path('docs/revision_plan/task_board.csv'),
}
optional_final_inputs = {
    'paired_raw_mamba': revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
    'external_protocol_gate_summary': revision_root / 'metrics' / 'external_protocol_gate_summary.csv',
    'representation_evidence_status': revision_root / 'metrics' / 'representation_evidence_status.json',
    'representation_probe_summary': revision_root / 'metrics' / 'representation_probe_summary.json',
    'representation_probe_table': revision_root / 'tables' / 'table_representation_probe.csv',
    'representation_cka_table': revision_root / 'tables' / 'table_representation_cka.csv',
    'representation_probe_manifest': revision_root / 'manifests' / 'representation_probe_manifest.json',
    'fewshot_ptbxl_summary': revision_root / 'metrics' / 'fewshot_ptbxl_summary.csv',
    'fewshot_ptbxl_table': revision_root / 'tables' / 'table_fewshot_ptbxl.csv',
    'fewshot_ptbxl_bootstrap': revision_root / 'metrics' / 'fewshot_ptbxl_bootstrap.json',
    'fewshot_ptbxl_manifest': revision_root / 'manifests' / 'fewshot_ptbxl_run_manifest.json',
    'robustness_multicomparator_summary': revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    'robustness_multicomparator_pairwise': revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    'robustness_multicomparator_table': revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    'robustness_multicomparator_manifest': revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
}
def collect_missing_final_inputs(verbose=True):
    missing = []
    for label, path in required_final_inputs.items():
        exists = path.exists()
        if verbose:
            print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')
        if not exists:
            missing.append(f'{label}={path}')
    return missing

missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    print('\nRequired inputs are missing from the active repo. Attempting verified Drive mirror restore before failing.')
    if 'MIRROR_REVISION_ROOT' not in globals():
        raise RuntimeError('MIRROR_REVISION_ROOT is undefined. Run the Setup cell first.')
    if not MIRROR_REVISION_ROOT.exists():
        raise FileNotFoundError(f'Mirror root does not exist: {MIRROR_REVISION_ROOT}')
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_required_restore.log',
    )
    print('\nRechecking required final evidence inputs after mirror restore:')
    missing_inputs = collect_missing_final_inputs(verbose=True)
    if missing_inputs:
        print('\nVerified mirror manifest did not restore every required input. Trying direct path fallback from Drive mirror/repo roots.')
        import shutil

        def _candidate_source_roots():
            roots = []
            if 'MIRROR_REVISION_ROOT' in globals():
                roots.append(('verified_mirror_path', Path(MIRROR_REVISION_ROOT)))
            drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
            roots.append(('drive_repo_reports', Path(drive_root) / 'ECG-RAMBA' / 'reports' / 'revision'))
            roots.append(('drive_revision_artifacts', Path(drive_root) / 'revision_artifacts' / 'reports' / 'revision'))
            return roots

        restored_direct, unresolved_direct = [], []
        for label, destination in required_final_inputs.items():
            destination = Path(destination)
            if destination.exists() and destination.stat().st_size > 0:
                continue
            try:
                rel_to_revision = destination.relative_to(revision_root)
            except ValueError:
                unresolved_direct.append(f'{label}={destination} (not under reports/revision; direct fallback skipped)')
                continue
            copied = False
            for source_label, source_root in _candidate_source_roots():
                source = source_root / rel_to_revision
                if source.exists() and source.stat().st_size > 0:
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, destination)
                    restored_direct.append(f'{label}: {source_label}:{source} -> {destination}')
                    copied = True
                    break
            if not copied:
                unresolved_direct.append(f'{label}={destination}')
        print(f'Direct final-input fallback restore: restored={len(restored_direct)} unresolved={len(unresolved_direct)}')
        for item in restored_direct:
            print(' - restored', item)
        for item in unresolved_direct:
            print(' - unresolved', item)
        print('\nRechecking required final evidence inputs after direct fallback restore:')
        missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    raise FileNotFoundError('Missing required final evidence inputs after mirror restore/direct fallback: ' + '; '.join(missing_inputs))

def restore_optional_final_inputs_from_drive():
    import shutil

    def _candidate_source_roots():
        roots = []
        if 'MIRROR_REVISION_ROOT' in globals():
            roots.append(('verified_mirror_path', Path(MIRROR_REVISION_ROOT)))
        drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
        roots.append(('drive_repo_reports', Path(drive_root) / 'ECG-RAMBA' / 'reports' / 'revision'))
        roots.append(('drive_revision_artifacts', Path(drive_root) / 'revision_artifacts' / 'reports' / 'revision'))
        return roots

    restored, still_missing = [], []
    for label, destination in optional_final_inputs.items():
        destination = Path(destination)
        if destination.exists() and destination.stat().st_size > 0:
            continue
        try:
            rel_to_revision = destination.relative_to(revision_root)
        except ValueError:
            still_missing.append(f'{label}={destination} (not under reports/revision)')
            continue
        copied = False
        for source_label, source_root in _candidate_source_roots():
            source = source_root / rel_to_revision
            if source.exists() and source.stat().st_size > 0:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, destination)
                restored.append(f'{label}: {source_label}:{source} -> {destination}')
                copied = True
                break
        if not copied:
            drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
            flat_source = Path(drive_root) / 'final_evidence_tables' / destination.name
            if flat_source.exists() and flat_source.stat().st_size > 0:
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(flat_source, destination)
                restored.append(f'{label}: final_evidence_tables:{flat_source} -> {destination}')
                copied = True
        if not copied:
            still_missing.append(f'{label}={destination}')
    print(f'Optional final-input path fallback restore: restored={len(restored)} still_missing={len(still_missing)}')
    for item in restored:
        print(' - restored', item)
    if still_missing:
        print('Optional inputs still absent after fallback; these remain deferred unless their notebook cells are run:')
        for item in still_missing:
            print(' - missing', item)


restore_optional_final_inputs_from_drive()

print('Optional final evidence inputs:')
for label, path in optional_final_inputs.items():
    exists = path.exists()
    print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')

# Guard against stale Colab code: if few-shot artifacts exist, the final generator
# must know how to ingest them into C06 and claim_guidance.
generator_path = Path('scripts/revision/13_final_evidence_matrix.py')
generator_source = generator_path.read_text(encoding='utf-8')
required_generator_tokens = [
    'def summarize_fewshot',
    'fewshot_summary = summarize_fewshot',
    '"fewshot_summary": fewshot_summary',
    '"fewshot": (',
]
missing_generator_tokens = [token for token in required_generator_tokens if token not in generator_source]
if missing_generator_tokens:
    raise RuntimeError(
        'Final evidence generator is stale and cannot ingest PTB-XL few-shot artifacts. '
        'Pull latest main / rerun Notebook 00 setup, then rerun Notebook 07. Missing tokens: '
        + ', '.join(missing_generator_tokens)
    )
print('Final evidence generator few-shot support: OK')

fewshot_preflight_paths = [
    optional_final_inputs['fewshot_ptbxl_summary'],
    optional_final_inputs['fewshot_ptbxl_table'],
    optional_final_inputs['fewshot_ptbxl_bootstrap'],
    optional_final_inputs['fewshot_ptbxl_manifest'],
]
fewshot_artifacts_present = all(path.exists() and path.stat().st_size > 0 for path in fewshot_preflight_paths)
print('Few-shot artifact set complete:', fewshot_artifacts_present)
if fewshot_artifacts_present:
    fewshot_manifest = json.loads(optional_final_inputs['fewshot_ptbxl_manifest'].read_text(encoding='utf-8'))
    print('Few-shot manifest status:', fewshot_manifest.get('status'))
    if fewshot_manifest.get('status') != 'complete':
        raise RuntimeError('Few-shot manifest exists but is not complete; rerun Notebook 02 few-shot cell or remove stale artifact.')

paired_resnet = json.loads(required_final_inputs['paired_resnet'].read_text(encoding='utf-8'))
resnet_metrics = paired_resnet.get('metrics', {})
print('Paired ResNet interpretations:')
for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
    row = resnet_metrics.get(metric, {})
    print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")

paired_raw_path = optional_final_inputs['paired_raw_mamba']
if paired_raw_path.exists():
    paired_raw = json.loads(paired_raw_path.read_text(encoding='utf-8'))
    raw_metrics = paired_raw.get('metrics', {})
    print('Paired Raw Mamba interpretations:')
    for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
        row = raw_metrics.get(metric, {})
        print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")
else:
    print('Paired Raw Mamba optional input is absent; final matrix will keep Raw Mamba incomplete until Notebook 04 produces it.')

expected_external_datasets = ['ptbxl', 'georgia', 'cpsc2021']
external_gate_path = optional_final_inputs['external_protocol_gate_summary']
if external_gate_path.exists():
    external_gate = pd.read_csv(external_gate_path)
    external_gate_columns = [
        'dataset',
        'status',
        'protocol_gate_passed',
        'manuscript_ready',
        'issue_count',
        'reused_existing',
        'gate_cache_key',
        'prediction_sha256',
        'slice_prediction_sha256',
        'gate_json',
        'gate_manifest',
    ]
    display_columns = [col for col in external_gate_columns if col in external_gate.columns]
    print('External protocol gate summary:')
    display(external_gate[display_columns])
    gate_pass_mask = external_gate.get('protocol_gate_passed', pd.Series(False, index=external_gate.index)).astype(str).str.lower().isin({'true', '1', 'yes'})
    present_datasets = sorted(external_gate['dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    passed_datasets = sorted(external_gate.loc[gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    blocked_datasets = sorted(external_gate.loc[~gate_pass_mask, 'dataset'].astype(str).str.lower().tolist()) if 'dataset' in external_gate.columns else []
    deferred_datasets = [dataset for dataset in expected_external_datasets if dataset not in present_datasets]
    print('External gate passed datasets:', ', '.join(passed_datasets) if passed_datasets else 'none')
    print('External gate blocked datasets in summary:', ', '.join(blocked_datasets) if blocked_datasets else 'none')
    print('External deferred datasets absent from gate summary:', ', '.join(deferred_datasets) if deferred_datasets else 'none')
    print('External gate supporting artifacts:')
    for dataset in sorted(external_gate['dataset'].astype(str).tolist()) if 'dataset' in external_gate.columns else []:
        supporting = {
            'gate_json': revision_root / 'metrics' / f'external_{dataset}_protocol_gate.json',
            'label_mapping': revision_root / 'tables' / f'table_external_{dataset}_label_mapping.csv',
            'metrics_table': revision_root / 'tables' / f'table_external_{dataset}_metrics.csv',
            'gate_manifest': revision_root / 'manifests' / f'external_{dataset}_protocol_gate_manifest.json',
        }
        for label, path in supporting.items():
            print(f'  {dataset:8} {label:14}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
else:
    print('External protocol gate summary absent; external datasets remain experimental/deferred in final evidence.')


## Final Evidence Matrix

Build claim-level evidence, blocker status, robustness claim rows, and reviewer-safe wording from frozen artifacts.

In [ ]:
import pandas as pd

run(
    'python -u scripts/revision/13_final_evidence_matrix.py --strict',
    log_path='reports/revision/logs/final_evidence_matrix.log',
)

matrix = pd.read_csv('reports/revision/tables/table_final_evidence_matrix.csv')
safe = pd.read_csv('reports/revision/tables/table_final_safe_wording.csv')
robust = pd.read_csv('reports/revision/tables/table_final_robustness_claims.csv')
blockers = pd.read_csv('reports/revision/tables/table_final_blocker_status.csv')
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))

if 'fewshot_artifacts_present' in globals() and fewshot_artifacts_present:
    fewshot_payload = payload.get('fewshot_summary', {})
    if fewshot_payload.get('complete') is not True:
        raise RuntimeError(
            'Few-shot artifacts are present, but final_evidence_matrix.json did not ingest them as complete. '
            'Check scripts/revision/13_final_evidence_matrix.py and rerun this cell.'
        )
    c06_rows = matrix.loc[matrix['claim_id'] == 'C06']
    c06_key_numbers = str(c06_rows.iloc[0]['key_numbers']) if len(c06_rows) else ''
    if 'fewshot_status=complete' not in c06_key_numbers:
        raise RuntimeError('C06 key_numbers does not include fewshot_status=complete; final generator is stale or artifact ingest failed.')
    if 'fewshot' not in payload.get('claim_guidance', {}):
        raise RuntimeError('claim_guidance lacks the fewshot wording block; final generator is stale.')
    print('Few-shot final evidence ingest: OK')
    print('Few-shot summary:', fewshot_payload.get('key_numbers'))
else:
    print('Few-shot final evidence ingest: deferred because complete few-shot artifacts are absent.')

print('Evidence matrix:')
display(matrix[['claim_id', 'claim_topic', 'evidence_status', 'key_numbers', 'blocker']])
print('Safe wording:')
display(safe[['claim_id', 'evidence_status', 'safe_wording']])
print('Robustness claim rows:', len(robust))
display(robust[['stress_test', 'metric', 'degradation_interpretation', 'stressed_performance_interpretation']])
print('A0 blocker status:')
display(blockers[['blocker_id', 'resolution_status', 'restriction']])


## Validation Gate

Fail if the final evidence matrix is incomplete or if blocked/limited claims are accidentally marked as fully supported.

In [ ]:
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('reports/revision/manifests/final_evidence_matrix_manifest.json').read_text(encoding='utf-8'))
required_outputs = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError('Missing final evidence outputs: ' + '; '.join(missing))
if payload.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence matrix is not ready for rebuttal use. Inspect unresolved_blockers/contract_issues.')
if payload.get('all_claims_supported') is True:
    raise RuntimeError('Unexpected all_claims_supported=True; blocked/limited claims must remain explicit.')
if len(payload.get('evidence_matrix', [])) != 6:
    raise RuntimeError(f"Final evidence matrix should have 6 claim rows, found {len(payload.get('evidence_matrix', []))}")
if len(payload.get('robustness_claims', [])) != 30:
    raise RuntimeError(f"Final robustness claim table should have 30 rows, found {len(payload.get('robustness_claims', []))}")
if payload.get('missing_inputs'):
    raise RuntimeError('Final evidence matrix reports missing inputs: ' + '; '.join(payload.get('missing_inputs', [])))
if manifest.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence manifest does not mark rebuttal readiness.')
matrix_by_claim = {row.get('claim_id'): row for row in payload.get('evidence_matrix', [])}
c01 = matrix_by_claim.get('C01', {})
c02 = matrix_by_claim.get('C02', {})
c03 = matrix_by_claim.get('C03', {})
c04 = matrix_by_claim.get('C04', {})
c06 = matrix_by_claim.get('C06', {})
if c01.get('source_claim_status') == 'blocked_fair_baselines_missing':
    raise RuntimeError('C01 source_claim_status is stale after Raw Mamba completion.')
for stale_phrase in ['external-transfer, or efficiency evidence', 'If domain AUC is high', 'Robustness rows=']:
    blob = '\n'.join(str(row) for row in payload.get('evidence_matrix', []))
    if stale_phrase in blob:
        raise RuntimeError(f'Stale final wording remains: {stale_phrase}')
if 'ResNet1D/CNN is stronger on PR-AUC, ROC-AUC, F1, Brier, and ECE' not in c02.get('safe_wording', ''):
    raise RuntimeError('C02 safe wording must state the ResNet comparator-specific result.')
if 'near-perfect HRV domain classifier' not in c03.get('safe_wording', ''):
    raise RuntimeError('C03 safe wording must state observed HRV domain sensitivity.')
c04_status = c04.get('evidence_status', '')
c04_numbers = c04.get('key_numbers', '')
c04_safe = c04.get('safe_wording', '')
if c04_status == 'complete_probe_available_with_disentanglement_limitation':
    if 'Probe/CKA complete' not in c04_numbers or 'CKA morphology/rhythm=' not in c04_numbers:
        raise RuntimeError('C04 key_numbers must summarize completed probe/CKA evidence.')
    if 'do not support label-aligned morphology-rhythm disentanglement' not in c04_safe:
        raise RuntimeError('C04 safe wording must forbid disentanglement overclaiming after probe completion.')
elif 'representation separation remains unproven' not in c04_numbers:
    raise RuntimeError('C04 key_numbers must either summarize probe/CKA evidence or keep representation separation unproven.')
external_gate_summary = Path('reports/revision/metrics/external_protocol_gate_summary.csv')
if external_gate_summary.exists():
    gate_df = pd.read_csv(external_gate_summary)
    gate_pass_mask = gate_df.get('protocol_gate_passed', pd.Series(False, index=gate_df.index)).astype(str).str.lower().isin({'true', '1', 'yes'})
    passed_gate_datasets = set(gate_df.loc[gate_pass_mask, 'dataset'].astype(str).str.lower()) if 'dataset' in gate_df.columns else set()
    c06_key_numbers = c06.get('key_numbers', '')
    if gate_pass_mask.any() and 'protocol_gated' not in c06.get('evidence_status', ''):
        raise RuntimeError('C06 evidence_status must reflect protocol-gated external datasets after a gate pass.')
    if passed_gate_datasets == {'ptbxl'} and 'external_gate_deferred=georgia,cpsc2021' not in c06_key_numbers:
        raise RuntimeError('C06 key_numbers must explicitly mark Georgia/CPSC2021 deferred for the stable manuscript plan.')
    if 'unqualified external-transfer claim' not in c06.get('safe_wording', ''):
        raise RuntimeError('C06 safe wording must explicitly restrict unqualified external-transfer claims.')
print('Final evidence matrix validated.')
print('All claims supported:', payload.get('all_claims_supported'))
print('Contract issues:', payload.get('contract_issues'))


## Inventory And Stable Mirror

Refresh the artifact manifest and publish final evidence outputs back to Drive.

In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/final_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/final_evidence_mirror_publish.log',
)


## Convenience Drive Copies

Copy final reviewer tables to a short Drive path for manual download/opening. These are convenience copies; the canonical artifacts remain under `reports/revision/` and the verified mirror.

In [ ]:
import shutil

FINAL_TABLE_EXPORT_DIR = DRIVE_ROOT / 'final_evidence_tables'
FINAL_TABLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_table_sources = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
optional_table_sources = [
    Path('reports/revision/metrics/external_protocol_gate_summary.csv'),
    Path('reports/revision/metrics/external_ptbxl_protocol_gate.json'),
    Path('reports/revision/tables/table_external_ptbxl_label_mapping.csv'),
    Path('reports/revision/tables/table_external_ptbxl_metrics.csv'),
    Path('reports/revision/manifests/external_ptbxl_protocol_gate_manifest.json'),
    Path('reports/revision/metrics/external_georgia_protocol_gate.json'),
    Path('reports/revision/tables/table_external_georgia_label_mapping.csv'),
    Path('reports/revision/tables/table_external_georgia_metrics.csv'),
    Path('reports/revision/manifests/external_georgia_protocol_gate_manifest.json'),
    Path('reports/revision/metrics/external_cpsc2021_protocol_gate.json'),
    Path('reports/revision/tables/table_external_cpsc2021_label_mapping.csv'),
    Path('reports/revision/tables/table_external_cpsc2021_metrics.csv'),
    Path('reports/revision/manifests/external_cpsc2021_protocol_gate_manifest.json'),
    Path('reports/revision/metrics/representation_evidence_status.json'),
    Path('reports/revision/metrics/representation_probe_summary.json'),
    Path('reports/revision/tables/table_representation_probe.csv'),
    Path('reports/revision/tables/table_representation_cka.csv'),
    Path('reports/revision/manifests/representation_probe_manifest.json'),
    Path('reports/revision/metrics/fewshot_ptbxl_summary.csv'),
    Path('reports/revision/tables/table_fewshot_ptbxl.csv'),
    Path('reports/revision/metrics/fewshot_ptbxl_bootstrap.json'),
    Path('reports/revision/manifests/fewshot_ptbxl_run_manifest.json'),
    Path('reports/revision/metrics/robustness_multicomparator_summary.csv'),
    Path('reports/revision/metrics/robustness_multicomparator_pairwise.json'),
    Path('reports/revision/tables/table_robustness_multicomparator.csv'),
    Path('reports/revision/manifests/robustness_multicomparator_manifest.json'),
]
for source in final_table_sources:
    if not source.exists():
        raise FileNotFoundError(source)
    destination = FINAL_TABLE_EXPORT_DIR / source.name
    shutil.copy2(source, destination)
    print(f'Copied: {destination} size={destination.stat().st_size}')

print('Optional convenience copies:')
for source in optional_table_sources:
    if source.exists():
        destination = FINAL_TABLE_EXPORT_DIR / source.name
        shutil.copy2(source, destination)
        print(f'Copied optional: {destination} size={destination.stat().st_size}')
    else:
        print(f'Skipped optional missing: {source}')

readme = FINAL_TABLE_EXPORT_DIR / 'README.md'
readme.write_text(
    '# ECG-RAMBA final evidence tables\n\n'
    'Convenience copies generated by notebooks/07_results_freeze.ipynb.\n\n'
    '- table_final_evidence_matrix.csv: claim-level evidence status and key numbers.\n'
    '- table_final_safe_wording.csv: reviewer/manuscript-safe wording.\n'
    '- table_final_blocker_status.csv: A0 blocker decisions and restrictions.\n'
    '- table_final_robustness_claims.csv: stress/metric-specific robustness CI interpretations.\n'
    '- final_evidence_matrix.json: full machine-readable payload.\n'
    '- external_protocol_gate_summary.csv and table_external_* files: optional protocol-gated external evidence copied when present. For the stable manuscript package, PTB-XL is the mapped-task external gate and Georgia/CPSC2021 remain deferred unless their own gates pass.\n'
    '- representation_probe_summary.json, table_representation_probe.csv, and table_representation_cka.csv: optional representation audit copied when present; use only as suggestive/limitation evidence, not proof of disentanglement.\n'
    '- fewshot_ptbxl_* files: optional gated score-calibration sensitivity analysis, not model-weight updating.\n'
    '- robustness_multicomparator_* files: optional ledger for robustness beyond MiniRocket; missing ResNet/Raw-Mamba stress predictions remain blocked rows.\n\n'
    'Canonical verified artifacts remain under reports/revision and the revision_artifacts mirror.\n',
    encoding='utf-8',
)
print('Convenience export dir:', FINAL_TABLE_EXPORT_DIR)


## Output Summary

In [ ]:
expected = [
    'reports/revision/metrics/final_evidence_matrix.json',
    'reports/revision/tables/table_final_evidence_matrix.csv',
    'reports/revision/tables/table_final_safe_wording.csv',
    'reports/revision/tables/table_final_blocker_status.csv',
    'reports/revision/tables/table_final_robustness_claims.csv',
    'reports/revision/manifests/final_evidence_matrix_manifest.json',
    'reports/revision/manifests/artifacts_manifest.json',
    'reports/revision/manifests/artifacts_manifest.csv',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_evidence_matrix.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_safe_wording.csv'),
]
for item in expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

optional_expected = [
    'reports/revision/metrics/paired_full_vs_raw_mamba_comparison.json',
    'reports/revision/tables/table_paired_full_vs_raw_mamba.csv',
    'reports/revision/manifests/paired_full_vs_raw_mamba_manifest.json',
    'reports/revision/metrics/external_protocol_gate_summary.csv',
    'reports/revision/metrics/external_ptbxl_protocol_gate.json',
    'reports/revision/tables/table_external_ptbxl_label_mapping.csv',
    'reports/revision/tables/table_external_ptbxl_metrics.csv',
    'reports/revision/manifests/external_ptbxl_protocol_gate_manifest.json',
    'reports/revision/metrics/external_georgia_protocol_gate.json',
    'reports/revision/tables/table_external_georgia_label_mapping.csv',
    'reports/revision/tables/table_external_georgia_metrics.csv',
    'reports/revision/manifests/external_georgia_protocol_gate_manifest.json',
    'reports/revision/metrics/external_cpsc2021_protocol_gate.json',
    'reports/revision/tables/table_external_cpsc2021_label_mapping.csv',
    'reports/revision/tables/table_external_cpsc2021_metrics.csv',
    'reports/revision/manifests/external_cpsc2021_protocol_gate_manifest.json',
    'reports/revision/metrics/representation_evidence_status.json',
    'reports/revision/metrics/representation_probe_summary.json',
    'reports/revision/tables/table_representation_probe.csv',
    'reports/revision/tables/table_representation_cka.csv',
    'reports/revision/manifests/representation_probe_manifest.json',
    'reports/revision/metrics/fewshot_ptbxl_summary.csv',
    'reports/revision/tables/table_fewshot_ptbxl.csv',
    'reports/revision/metrics/fewshot_ptbxl_bootstrap.json',
    'reports/revision/manifests/fewshot_ptbxl_run_manifest.json',
    'reports/revision/metrics/robustness_multicomparator_summary.csv',
    'reports/revision/metrics/robustness_multicomparator_pairwise.json',
    'reports/revision/tables/table_robustness_multicomparator.csv',
    'reports/revision/manifests/robustness_multicomparator_manifest.json',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'external_protocol_gate_summary.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_external_ptbxl_label_mapping.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_external_ptbxl_metrics.csv'),
]
print('\nOptional evidence artifacts:')
for item in optional_expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
print('\nFinal guidance:')
for key, value in payload.get('claim_guidance', {}).items():
    print(f'- {key}: {value}')
print('\nFew-shot final summary:')
print(payload.get('fewshot_summary', {}))
print('\nConvenience Drive folder:', DRIVE_ROOT / 'final_evidence_tables')
print('Notebook 07 complete. Use table_final_evidence_matrix.csv and table_final_safe_wording.csv as the manuscript/rebuttal source of truth.')
